# ⚔️ B2　Boss 戰：健康管理小幫手
**Python 冒險之旅 2026**　｜　Day 2（08/30 日）🌴 文字之島　｜　Boss 戰　｜　🏅 200 XP

📖 對應教科書：第 3–4 章綜合


### 🎯 這一關你會學到
- 輸入 → 計算 → 判斷 → 格式化輸出的完整流程

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "B2"
_SALT = "python-quest-2026-datama"
_TASKS = ["B2-1", "B2-2", "B2-3", "B2-4", "B2-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B2_1(run):
    out, ns = run("170", "66")
    if abs(ns.get("bmi", 0) - 22.8374) > 0.01: return (False, "170cm、66kg 的 BMI 約 22.8（身高要除以 100 換成公尺）。")
    return ("22.8" in out, "要用 {bmi:.1f} 顯示小數 1 位。")
任務定義("B2-1", _check_B2_1, 提示="bmi = w / (h / 100) ** 2。")

def _check_B2_2(run):
    for h, w, g in [("170", "50", "過輕"), ("170", "66", "正常"), ("170", "75", "過重"), ("170", "90", "肥胖"), ("160", "61.44", "正常")]:
        out, ns = run(h, w)
        if not 出現(out, "體位：" + g): return (False, f"{h}cm/{w}kg 應該是 {g}。注意邊界 24 與 27。")
    return True
任務定義("B2-2", _check_B2_2, 提示="elif bmi < 24: 正常；elif bmi < 27: 過重；else: 肥胖。")

def _check_B2_3(run):
    out, ns = run("66", "25")
    if ns.get("water") != 1980: return (False, "66 公斤的飲水量應該是 1980（要用 int() 取整數）。")
    if (ns.get("low"), ns.get("high")) != (117, 156): return (False, "25 歲的心率區間應該是 117 ～ 156。")
    return 出現(out, "1980", "117", "156")
任務定義("B2-3", _check_B2_3, 提示="water = int(w * 30)；low = int((220 - age) * 0.6)。")

def _check_B2_4(run):
    out, ns = run("0", "66")
    if not 出現(out, "輸入錯誤"): return (False, "身高 0 要顯示輸入錯誤。")
    out, ns = run("170", "-5")
    if not 出現(out, "輸入錯誤"): return (False, "體重 -5 要顯示輸入錯誤。")
    out, ns = run("170", "66")
    return (出現(out, "22.8") and not 出現(out, "輸入錯誤"), "正常輸入要顯示 BMI。")
任務定義("B2-4", _check_B2_4, 提示="條件：h <= 0 or w <= 0。")

def _check_B2_5(run):
    out, ns = run("小明", "170", "66", "25")
    need = ["小明的健康報告", "BMI：22.8（正常）", "每日飲水：1980毫升", "運動心率：117～156"]
    missing = [n for n in need if not 出現(out, n)]
    if missing: return (False, f"報告少了：{missing}")
    out2, ns2 = run("阿華", "160", "80", "40")
    return (出現(out2, "阿華", "31.2（肥胖）", "2400", "108～144"), "換一組資料（阿華 160/80/40）也要正確。")
任務定義("B2-5", _check_B2_5, 提示="把 B2-1 到 B2-3 的程式組合起來，最後用 f-string 印出四行。")


## ⚔️ Boss 登場：健康管理小幫手
你要打造一個健康小幫手：輸入身高、體重、年齡，算出 BMI、判斷體位、給出每日飲水量與運動心率建議，最後印出漂亮的報告。

| 規則 | 說明 |
|---|---|
| BMI | 體重(kg) ÷ 身高(m)² |
| 體位 | BMI < 18.5 過輕；18.5 ≤ BMI < 24 正常；24 ≤ BMI < 27 過重；≥ 27 肥胖 |
| 飲水量 | 體重 × 30 毫升 |
| 運動心率區間 | (220 − 年齡) × 0.6 ～ (220 − 年齡) × 0.8（取整數） |

> 用到：input、型別轉換、運算子、f-string 格式化、if/elif/else。

### 🎯 任務 B2-1　BMI 計算

讀取身高（公分）與體重（公斤），算出 `bmi` 並印出 `BMI = 22.9`（小數 1 位）。注意身高要先換成公尺。

In [ ]:
# 🎯 任務 B2-1　BMI 計算（請保留這一行）
h = float(input("身高(公分)："))
w = float(input("體重(公斤)："))
bmi = ???
print(f"BMI = {bmi:???}")

In [ ]:
檢查("B2-1")   # ◀ 執行這一格，看看任務 B2-1 有沒有過關

### 🎯 任務 B2-2　體位判斷

承上，依 BMI 印出體位：`過輕`、`正常`、`過重`、`肥胖`（格式：`體位：正常`）。

In [ ]:
# 🎯 任務 B2-2　體位判斷（請保留這一行）
h = float(input("身高(公分)："))
w = float(input("體重(公斤)："))
bmi = w / (h / 100) ** 2
if bmi < 18.5:
    status = "過輕"
# 補上其餘判斷
print(f"BMI = {bmi:.1f}，體位：{status}")

In [ ]:
檢查("B2-2")   # ◀ 執行這一格，看看任務 B2-2 有沒有過關

### 🎯 任務 B2-3　飲水與心率建議

讀取體重與年齡，印出兩行：`每日建議飲水量：1980 毫升`（體重 × 30，整數）與 `運動心率區間：117 ～ 156`（(220−年齡)×0.6 與 ×0.8，取整數）。

In [ ]:
# 🎯 任務 B2-3　飲水與心率建議（請保留這一行）
w = float(input("體重(公斤)："))
age = int(input("年齡："))
water = ???
low = ???
high = ???
print(f"每日建議飲水量：{water} 毫升")
print(f"運動心率區間：{low} ～ {high}")

In [ ]:
檢查("B2-3")   # ◀ 執行這一格，看看任務 B2-3 有沒有過關

### 🎯 任務 B2-4　輸入防呆

讀取身高與體重，如果任一個 **小於或等於 0** 就印出 `輸入錯誤：數值必須大於 0`，否則印出 BMI（小數 1 位）。

In [ ]:
# 🎯 任務 B2-4　輸入防呆（請保留這一行）
h = float(input("身高(公分)："))
w = float(input("體重(公斤)："))
if ???:
    print("輸入錯誤：數值必須大於 0")
else:
    bmi = w / (h / 100) ** 2
    print(f"BMI = {bmi:.1f}")

In [ ]:
檢查("B2-4")   # ◀ 執行這一格，看看任務 B2-4 有沒有過關

### 🎯 任務 B2-5　完整健康報告

整合以上：讀取姓名、身高、體重、年齡，印出下面格式的報告（數字以 小明、170、66、25 為例）：

```
===== 小明 的健康報告 =====
BMI：22.8（正常）
每日飲水：1980 毫升
運動心率：117 ～ 156
```

In [ ]:
# 🎯 任務 B2-5　完整健康報告（請保留這一行）
name = input("姓名：")
h = float(input("身高(公分)："))
w = float(input("體重(公斤)："))
age = int(input("年齡："))
# 計算 bmi、status、water、low、high，然後印出報告

In [ ]:
檢查("B2-5")   # ◀ 執行這一格，看看任務 B2-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 加上「理想體重」= 22 × 身高(m)²，並印出「距離理想體重還差 x 公斤」。
2. 如果年齡小於 18，改印「請參考兒童生長曲線」而不是 BMI 判斷。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🔁 L06 迴圈 for / while** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L06_loops.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/